In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd
import cartopy.util
import xesmf as xe

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Load data

### CESM2

In [ ]:
## load spatial data
_, anom = src.utils.load_consolidated()

## trim to desired vars
anom = anom[["sst", "pr", "ssh", "sst_comp", "pr_comp", "ssh_comp"]]

### Reference data

In [ ]:
## reference data
REF_FP = pathlib.Path("/glade/campaign/cgd/cas/asphilli/CVDP-OBS/")
pr_ref = xr.open_dataset(REF_FP / "gpcp.mon.mean.197901-202403.nc")["precip"]
sst_ref = xr.open_dataset(REF_FP / "ersstv5.185401-202412.nc")["ssta"]

## assign time coord
sst_ref["time"] = xr.date_range(start="1854-01", end="2024-12", freq="MS")

### Trim to overlapping period

In [ ]:
## specify index
time_idx = dict(time=slice("1958-01", "2024-06"))

## subset in time
sst_ref = sst_ref.sel(time_idx)
pr_ref = pr_ref.sel(time_idx)
anom = anom.sel(time_idx)

### resample to seasonal

In [ ]:
## func to resample with
resample = lambda x: x.resample({"time": "QS-NOV"}).mean("time")

## specify month for averaging
MONTH = 11

## resample obs
sst_ref = resample(sst_ref)
pr_ref = resample(sst_ref)

## resample model
anom_ = copy.deepcopy(anom[["sst_comp"]])
for v in tqdm.tqdm(["sst"]):
    anom_[v] = resample(anom[v])

### Regrid to match

In [ ]:
## rename spatial coords
if "lat" in pr_ref.coords:
    coord_dict = dict(lat="latitude", lon="longitude")
    pr_ref = pr_ref.rename(coord_dict)
    sst_ref = sst_ref.rename(coord_dict)

## regrid
grid = anom["sst_comp"].isel(mode=0)
sst_regridder = xe.Regridder(sst_ref, grid, "bilinear")
sst_ref = sst_regridder(sst_ref)
pr_regridder = xe.Regridder(pr_ref, grid, "bilinear")
pr_ref = pr_regridder(pr_ref)

## Make composites

In [ ]:
## specify index fn
idx_fn = src.utils.get_nino3

## Get DJF
anom_ = anom_.sel(time=anom_.time.dt.month == MONTH)
sst_ref = sst_ref.sel(time=sst_ref.time.dt.month == MONTH)

## Get index
idx_ref = idx_fn(sst_ref)
idx_anom_ = src.utils.reconstruct_wrapper(anom_, idx_fn)["sst"]

In [ ]:
def get_warm_composite(data, idx, q=0.75):
    """get warm composite"""

    ## get masked data
    data_ = data.where(idx > idx.quantile(q=q, dim="time"))

    return data_.mean("time").drop_vars("quantile")


def get_cold_composite(data, idx, q=0.75):
    """get warm composite"""

    ## get masked data
    data_ = data.where(idx < idx.quantile(q=(1 - q), dim="time"))

    return data_.mean("time").drop_vars("quantile")


def get_composites(data, idx, q=0.75):
    """get composites"""

    kwargs = dict(data=data, idx=idx, q=q)
    comps = xr.merge(
        [
            get_warm_composite(**kwargs).rename("warm"),
            get_cold_composite(**kwargs).rename("cold"),
        ],
    )

    return comps


## get composites
q = 0.9
comps_ref = get_composites(sst_ref, idx=idx_ref, q=q)
comps_anom_ = get_composites(anom_["sst"], idx=idx_anom_, q=q).mean("member")

Reconstruct CESM2

In [ ]:
comps_anom = copy.deepcopy(comps_anom_)

for n in ["warm", "cold"]:
    comps_anom[n] = src.utils.reconstruct_fn(
        scores=comps_anom[n],
        components=anom["sst_comp"],
        fn=lambda x: x,
    )

## drop mode coordinate
comps_anom = comps_anom.drop_vars("mode")

## normalize by El Niño
get_scale = lambda x: np.abs(src.utils.get_nino3(x["warm"]))
scale_ref = get_scale(comps_ref)
scale_anom = get_scale(comps_anom)
comps_ref_norm = comps_ref / scale_ref
comps_anom_norm = comps_anom / scale_anom

## Plot

### funcs

In [ ]:
import cartopy.crs as ccrs


def plot_sst(ax, data, amp=2):
    """plot data on ax"""

    ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(amp, amp / 10),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return

In [ ]:
## specify lons/lats for E/W boxes
LONS_E = [240, 280]
LONS_W = [150, 190]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)

### Spatial plot

In [ ]:
## specify amp
AMP = 1.25

fig = plt.figure(figsize=(10, 4), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=20, lon_range=(130, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)

plot_sst(axs[0, 0], comps_ref_norm["warm"], amp=AMP)
plot_sst(axs[0, 1], comps_anom_norm["warm"], amp=AMP)

plot_sst(axs[1, 0], comps_ref_norm["cold"], amp=AMP)
plot_sst(axs[1, 1], comps_anom_norm["cold"], amp=AMP)

plot_sst(axs[2, 0], comps_ref_norm["warm"] + comps_ref_norm["cold"], amp=0.75)
plot_sst(axs[2, 1], comps_anom_norm["warm"] + comps_anom_norm["cold"], amp=0.75)

## plot boxes
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

plt.show()

### Scatter plot

#### reconstruct indices

In [ ]:
## specify x func
x_fn = src.utils.get_nino34

## get scale for CESM for comparable plotting
scale_ratio = scale_ref / scale_anom

## compute indices
x_anom = scale_ratio * src.utils.reconstruct_wrapper(anom_, x_fn)["sst"]
Te_anom = scale_ratio * src.utils.reconstruct_wrapper(anom_, get_e)["sst"]
Tw_anom = scale_ratio * src.utils.reconstruct_wrapper(anom_, get_w)["sst"]

## get grad (already scaled)
dTdx_anom = Te_anom - Tw_anom

## compute PDFs

# get indices
pdf_idx_fn = src.utils.get_nino3
# pdf_idx_fn = get_dx
pdf_idx_anom = scale_ratio * src.utils.reconstruct_wrapper(anom_, pdf_idx_fn)["sst"]
pdf_idx_ref = pdf_idx_fn(sst_ref)

## compute
bins = np.arange(-4.25, 4.75, 0.5)
pdf_anom, _ = src.utils.get_empirical_pdf(pdf_idx_anom, bins)
pdf_ref, _ = src.utils.get_empirical_pdf(pdf_idx_ref, bins)

## args for scatter plot
kwargs = dict(s=25)

## make scatter plot
fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

## plot data
axs[0].scatter(x_fn(sst_ref), get_dx(sst_ref), s=25, c="gray")
axs[1].scatter(x_anom, dTdx_anom, alpha=0.3, s=1, c="gray")

## guidelines
for ax in axs[:2]:
    ax.axhline(0, ls="--", c="k", lw=0.5)
    ax.axvline(0, ls="--", c="k", lw=0.5)
    ax.set_xticks([-3, 0, 3])
    zz = np.linspace(-1, 3.5)
    ax.plot(zz, zz, c="k", lw=0.8)
    ax.set_xlabel("Niño 3.4 (K)")

## pdfs
axs[-1].stairs(pdf_ref, bins, fill=sns.color_palette()[0], alpha=0.3, label="ERSST")
axs[-1].stairs(pdf_anom, bins, label="CESM2")
axs[-1].axvline(0, c="k", lw=0.8, ls="--")

## formatting
src.utils.set_lims(axs[:2])
axs[0].set_yticks([-1.5, 0, 1.5, 3])
axs[0].set_ylabel(r"$T_e-T_w$ (K)")
axs[1].set_yticks([])
axs[-1].legend()
axs[0].set_title("ERSST")
axs[1].set_title("CESM2")
axs[2].set_title("PDFs")
axs[2].set_xticks([-3, 0, 3])
axs[2].yaxis.set_label_position("right")
axs[2].yaxis.tick_right()
axs[2].set_yticks([0, 0.3, 0.6])
axs[2].set_xlim([-3.75, 3.75])
axs[2].set_xlabel("Niño 3 (K)")
axs[2].set_ylabel("Prob. density")

plt.show()

## Scratch

In [ ]:
## specify amp
AMP = 1.5

fig = plt.figure(figsize=(10, 4), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=20, lon_range=(120, 285))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)

plot_sst(axs[0, 0], comps_ref["warm"], amp=AMP)
plot_sst(axs[0, 1], comps_anom["warm"] - comps_ref["warm"], amp=AMP / 3)

plot_sst(axs[1, 0], comps_ref["cold"], amp=AMP)
plot_sst(axs[1, 1], comps_anom["cold"] - comps_ref["cold"], amp=AMP / 3)

plot_sst(axs[2, 0], comps_ref["warm"] + comps_ref["cold"], amp=0.75)
plot_sst(axs[2, 1], comps_anom["warm"] + comps_anom["cold"], amp=0.75)